
# Árboles de Decisión en Clasificación (scikit-learn)

**Objetivo.** Explorar y evaluar árboles de decisión:
- **Clasificación** con el dataset **Breast Cancer Wisconsin** (`sklearn.datasets.load_breast_cancer`).


### Descripción del conjunto de datos: Breast Cancer Wisconsin

Este conjunto de datos proviene del **Breast Cancer Wisconsin Diagnostic Dataset**, incluido en `scikit-learn`.  
Contiene mediciones obtenidas de imágenes digitalizadas de muestras de tejido mamario, con el objetivo de **clasificar los tumores como benignos o malignos**.

**Características principales:**

- **Número de muestras:** 569  
- **Número de variables:** 30 numéricas continuas  
- **Variable objetivo:** `target`  
  - `0` = maligno  
  - `1` = benigno  

**Variables explicativas:**  
Cada observación describe propiedades estadísticas de los núcleos celulares extraídos de una imagen digital.  
Las 30 características derivan de 10 medidas básicas (radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría, dimensión fractal), calculadas como:
- media (*mean*),  
- error estándar (*se*),  
- valor máximo (*worst*).

**Objetivo del modelo:**  
Predecir si una muestra es **maligna o benigna** a partir de estas características morfológicas, evaluando la capacidad del árbol de decisión para separar ambas clases de manera interpretable.





#### Preparación del entorno


In [ ]:

# Paquetes básicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve, validation_curve
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier,  plot_tree

# Datasets
from sklearn.datasets import load_breast_cancer

# Reproducibilidad
np.random.seed(42)



## Clasificación: Breast Cancer Wisconsin

**Tarea.** Diagnosticar tumores como *malignos* o *benignos*.


In [ ]:

# Carga del dataset
data_clf = load_breast_cancer()
X_clf = pd.DataFrame(data_clf.data, columns=data_clf.feature_names)
y_clf = pd.Series(data_clf.target, name='target')

X_clf.head()


#### División en Train-Test

Usa `train_test_split` para hacer la división del conjunto de datos entre entrenamiento y test.

Emplea el parámetro `stratify=y_clf` asegura que la **proporción de clases (benigno/maligno)** se mantenga igual en ambos subconjuntos, evitando sesgos por desbalanceo.

In [ ]:

# Partición train/test
# *************  TU CODIGO AQUI *********************************
Xc_train, Xc_test, yc_train, yc_test = 
# ***************************************************************
Xc_train.shape, Xc_test.shape, yc_train.mean(), yc_test.mean()



###  Baseline con validación cruzada

Usamos un árbol sin ajustar como referencia. Luego mediremos mejora al ajustar hiperparámetros.

En esta etapa se entrena un **árbol de decisión sin ajuste de hiperparámetros** para establecer una línea base (*baseline*) de rendimiento.

- Se usa **validación cruzada de 5 particiones (cv=5)** sobre el conjunto de entrenamiento para estimar la precisión media y su desviación estándar.  
- Dado que los árboles de decisión son insensibles al escalado de características, **no se aplica normalización ni estandarización**.  

Tras la validación, se entrena el modelo sobre **todo el conjunto de entrenamiento** y se evalúa su rendimiento en el **conjunto de prueba**.



In [ ]:

# Pipeline simple: (Árbol no requiere escalado, lo dejamos directo)
dtc_base = DecisionTreeClassifier(random_state=42)
cv_scores = cross_val_score(dtc_base, Xc_train, yc_train, cv=5, scoring='accuracy')
print('CV accuracy media:', cv_scores.mean().round(4), '±', cv_scores.std().round(4))

# Entrenar en train completo y evaluar en test
# *************  TU CODIGO AQUI *********************************

yc_pred = 

# ***************************************************************
print('Test accuracy:', accuracy_score(yc_test, yc_pred))
print('\nReporte de clasificación:\n', classification_report(yc_test, yc_pred, target_names=data_clf.target_names))


### Ajuste de hiperparámetros clave mediante Grid Search

En esta celda debes optimizar los **hiperparámetros clave** del árbol de decisión usando `GridSearchCV`, que evalúa combinaciones de parámetros mediante validación cruzada.

**Parámetros explorados:**
- `max_depth`: profundidad máxima del árbol. Controla el grado de ajuste; valores bajos reducen el sobreajuste.  
- `min_samples_split`: número mínimo de muestras necesarias para dividir un nodo.  
- `min_samples_leaf`: número mínimo de muestras requeridas en una hoja final.  
- `ccp_alpha`: parámetro de **poda por complejidad de coste**, que elimina ramas poco informativas.

El modelo con la mejor puntuación media de validación se guarda en `best_clf`.  
Luego se evalúa su rendimiento sobre el conjunto de prueba para obtener la **precisión final optimizada**.


In [ ]:
# Define la malla de parametros que vas a emplear

# Entrenar en train completo y evaluar en test
# *************  TU CODIGO AQUI *********************************
param_grid = {
    'max_depth': 
    'min_samples_split': 
    'min_samples_leaf':
    'ccp_alpha': 
}
# ***************************************************************

# Crea el clasificador base
# *************  TU CODIGO AQUI *********************************
dtc = 
# ***************************************************************

# Crea el objeto de la clase GridSearchCV y entenalo
# *************  TU CODIGO AQUI *********************************
grid = 
grid.fit(Xc_train, yc_train)
# ***************************************************************

print('Mejor score CV:', grid.best_score_.round(4))
print('Mejores params:', grid.best_params_)

# Guarda el mejor modelo 
# *************  TU CODIGO AQUI *********************************
best_clf =
# ***************************************************************

# Obtén las predicciones del mejor modelo sobre el conjunto de test 
# *************  TU CODIGO AQUI ********************************* 
yc_pred_best = 
# ***************************************************************

print('Test accuracy (mejor):', accuracy_score(yc_test, yc_pred_best))


### Matriz de confusión

La **matriz de confusión** resume los resultados de la clasificación mostrando cuántas observaciones fueron clasificadas correctamente o de forma errónea.

Cada fila representa la **clase real** y cada columna la **clase predicha** por el modelo:

- Los elementos en la **diagonal principal** indican predicciones correctas.  
- Los elementos **fuera de la diagonal** representan errores de clasificación (falsos positivos o falsos negativos).

Esta matriz permite evaluar de forma detallada el comportamiento del clasificador, especialmente en conjuntos con posibles desbalances entre clases.


In [ ]:

# Matriz de confusión
cm = confusion_matrix(yc_test, yc_pred_best)
cm_df = pd.DataFrame(cm, index=[f'Real_{c}' for c in data_clf.target_names],
                     columns=[f'Pred_{c}' for c in data_clf.target_names])
cm_df



### Visualización del árbol y relevancia de variables


In [ ]:

plt.figure(figsize=(12, 8))
plot_tree(best_clf, feature_names=data_clf.feature_names, class_names=data_clf.target_names, filled=False)
plt.title('Árbol de decisión óptimo (clasificación)')
plt.show()

# Importancias
# Haz una gráfica con las variables ordenadas según su importancia
# *************  TU CODIGO AQUI *********************************



# ***************************************************************



### EXTRA Curvas de validación.

Diagnóstico de sesgo-varianza.

Vamos a estudiar cómo varía el rendimiento del modelo al modificar un hiperparámetro clave.  
En este caso se estudia `max_depth`, que controla la **profundidad máxima del árbol** y, por tanto, el equilibrio entre sesgo y varianza.

- Para cada valor de `max_depth`, se calcula la **precisión media** en entrenamiento y validación cruzada (5 particiones).  
- Un valor de `max_depth` muy alto puede causar **sobreajuste** (alta precisión en entrenamiento, baja en validación).  
- Un valor demasiado bajo puede causar **infraajuste** (baja precisión en ambos conjuntos).

El gráfico muestra ambas curvas (`train` y `cv`) para ayudar a identificar la profundidad óptima que logra el mejor compromiso entre generalización y capacidad predictiva.


La función `validation_curve` hace barridos de un hiperparámetro y devuelve las puntuaciones de entrenamiento y validación para cada valor, usando validación cruzada.
Parametros clave:
 - `estimator`: modelo con API scikit-learn.
 - `X, y`: datos. `y` puede ser `None` en casos no supervisados.
 - `param_name`: nombre exacto del hiperparámetro, p. ej. "max_depth".
 - `param_range`: iterable de valores a probar.
 - `cv`: número de folds o splitter. Si `cv=None` ⇒ StratifiedKFold en clasificación y KFold en regresión.
 - `scoring`: métrica. 

Salida:
 - `train_scores`: array con la métrica en entrenamiento por valor y fold.
 - `test_scores`: array con la métrica en validación por valor y fold.





In [ ]:

# Curva de validación para max_depth
# Definimos un rango de valores para el parámetro a estudiar
param_range = [None, 2, 3, 4, 5, 6, 8, 10]

train_scores, test_scores = validation_curve(
    DecisionTreeClassifier(random_state=42),
    Xc_train, yc_train,
    param_name='max_depth', param_range=param_range,
    cv=5, scoring='accuracy', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure()
plt.plot(range(len(param_range)), train_mean, marker='o', label='train')
plt.plot(range(len(param_range)), test_mean, marker='s', label='cv')
plt.xticks(range(len(param_range)), [str(p) for p in param_range], rotation=0)
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.title('Curva de validación: max_depth (clasificación)')
plt.legend()
plt.show()
